# AMS-SkipGNN Kaggle Stage 4 — last run

Upload **this** notebook: `notebooks/ams_skipgnn_kaggle_stage4.ipynb`.

Use **GPU T4**, Internet **ON**, then **Save Version → Save & Run All**.

Clones `aryonmt/finalProject` branch **`feat/three-new-architectures`**. Override with `REPO_BRANCH`.

Default **`STAGE=4`**. This is the delivery run. It does **not** retrain Stage 1–3.

| STAGE | What runs |
| --- | --- |
| `0` | Smoke: DTI `--quick` |
| `4` (default) | Train `gat` / `3hop` / `contrastive` on **DDI, PPI, GDI** (skip a dataset/model if 3 seeds already exist), save embeddings, regenerate figures + t-SNE |

DTI already has the three new models from Stage 3. This notebook fills the remaining three datasets.

After the run, download **only** `/kaggle/working/ams_skipgnn_kaggle_bundle.zip` and drop it at the repo root (or send it back). Import folds CSVs, syncs `figures/`, and archives the executed notebook. Then the repo is delivery-ready.

DDI/PPI use **`--batch-size 1024`**. GDI uses **256** and a chunked GAT attention kernel so the skip-graph (~20k nodes) fits on a T4. Models run **one at a time**; already-finished seeds are skipped.

**Resume a failed Stage 4 version (keep DDI/PPI):**

1. Edit this notebook (use the latest file from GitHub).
2. **Add Input → Notebook Output Files** → the failed Stage 4 version.
3. **Save Version → Save & Run All** again. The next cell copies `results/` from that output, skips finished models, and trains GDI only.


Expected wall time on T4: remaining GDI work is the long part (GAT is the heavy one).


In [ ]:
import os, sys, platform, subprocess, shutil
from pathlib import Path

print('python', sys.version)
print('platform', platform.platform())
try:
    import torch
    print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('gpu', torch.cuda.get_device_name(0))
except Exception as e:
    print('torch import failed', e)

REPO = 'https://github.com/aryonmt/finalProject.git'
BRANCH = os.environ.get('REPO_BRANCH', 'feat/three-new-architectures')
WORK = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
ROOT = WORK / 'finalProject'
if (Path.cwd() / 'src' / 'models').exists():
    ROOT = Path.cwd()
    print('already in repo', ROOT)
else:
    if ROOT.exists():
        shutil.rmtree(ROOT)
    subprocess.check_call(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO, str(ROOT)])
    print('cloned', ROOT, 'branch', BRANCH)
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print('cwd', os.getcwd())


In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-e', '.', '-q'])
print('pip install -e . done')
subprocess.check_call([sys.executable, 'scripts/fetch_data.py'])
print('data fetch done')


In [ ]:
import subprocess, sys
rc = subprocess.call([sys.executable, '-m', 'pytest', '-q'])
print('pytest rc', rc)
assert rc == 0, 'smoke tests failed'


In [ ]:
import os, shutil, subprocess, sys, time
from pathlib import Path
import pandas as pd

stage = os.environ.get('STAGE', '4')
print('STAGE', stage, '(0=smoke, 4=new models on DDI/PPI/GDI)')
py = sys.executable
SEEDS = {7, 42, 123}
NEW_MODELS = ['gat', '3hop', 'contrastive']
DATASETS = ['DDI', 'PPI', 'GDI']


def _csv(ds):
    return Path(f'results/{ds}/benchmark.csv')


def _complete(ds, model):
    path = _csv(ds)
    if not path.exists():
        return False
    df = pd.read_csv(path)
    sub = df[df['model'].astype(str).str.lower() == model.lower()]
    if sub.empty or 'seed' not in sub.columns:
        return False
    have = set(int(s) for s in sub['seed'].dropna())
    if not SEEDS.issubset(have):
        return False
    if 'hard_auprc' in sub.columns:
        return bool(sub['hard_auprc'].notna().any())
    return True


BATCH = {'DDI': '1024', 'PPI': '1024', 'GDI': '256'}


def _ingest_previous_kaggle_output():
    """Copy results/figures from a previous Kaggle version attached as input."""
    roots = [Path('/kaggle/input'), Path('/kaggle/working')]
    copied = 0
    for root in roots:
        if not root.exists():
            continue
        for src in root.rglob('benchmark.csv'):
            ds = src.parent.name
            if ds not in {'DTI', 'DDI', 'PPI', 'GDI'}:
                continue
            if 'finalProject' in src.parts and src.resolve().is_relative_to(Path.cwd().resolve()):
                continue
            dest = Path('results') / ds
            dest.mkdir(parents=True, exist_ok=True)
            for item in src.parent.iterdir():
                target = dest / item.name
                if item.is_file():
                    if not target.exists() or item.stat().st_mtime >= target.stat().st_mtime:
                        shutil.copy2(item, target)
                        copied += 1
                elif item.is_dir() and item.name != '__pycache__':
                    shutil.copytree(item, target, dirs_exist_ok=True)
                    copied += 1
            print('ingested', src.parent, '->', dest, flush=True)
        for src in root.rglob('fig*.png'):
            if 'figures' not in src.parts:
                continue
            dest = Path('figures') / src.name
            dest.parent.mkdir(parents=True, exist_ok=True)
            if not dest.exists():
                shutil.copy2(src, dest)
                copied += 1
    print('ingest copied', copied, 'items', flush=True)


_ingest_previous_kaggle_output()


def _run(cmd):
    print('running', cmd, flush=True)
    subprocess.check_call(cmd)


t0 = time.time()
if stage == '0':
    _run([py, 'scripts/run_benchmark.py', '--dataset', 'DTI', '--models', 'gcn', '--quick'])
else:
    print('=== STAGE 4: gat / 3hop / contrastive on DDI, PPI, GDI ===')
    failed = []
    for ds in DATASETS:
        for m in NEW_MODELS:
            if _complete(ds, m):
                print(f'[{ds}] {m} already complete, skip')
                continue
            cmd = [
                py, 'scripts/run_benchmark.py',
                '--dataset', ds,
                '--models', m,
                '--batch-size', BATCH[ds],
                '--save-embeddings',
            ]
            try:
                _run(cmd)
            except subprocess.CalledProcessError as exc:
                print(f'FAILED {ds} {m} rc={exc.returncode}', flush=True)
                failed.append((ds, m, exc.returncode))
    print('training cell done in', round((time.time() - t0) / 60, 2), 'min')
    if failed:
        print('FAILED RUNS (continuing to pack whatever finished):', failed, flush=True)
    for ds in ['DTI', *DATASETS]:
        path = _csv(ds)
        if not path.exists():
            print(f'[{ds}] missing benchmark.csv')
            continue
        df = pd.read_csv(path)
        print(ds, sorted(df['model'].astype(str).str.lower().unique().tolist()))


In [ ]:
import json, os, shutil, subprocess, sys, zipfile
from datetime import datetime, timezone
from pathlib import Path

subprocess.call([sys.executable, 'scripts/plot_tsne.py'])
subprocess.call([sys.executable, 'scripts/make_figures.py', '--skip-tsne'])

repo = Path.cwd()
work = Path('/kaggle/working') if Path('/kaggle/working').exists() else repo
staging = work / '_kaggle_bundle_staging'
if staging.exists():
    shutil.rmtree(staging)
staging.mkdir(parents=True)


def copy_tree(src: Path, dest: Path) -> int:
    if not src.exists():
        return 0
    n = 0
    dest.mkdir(parents=True, exist_ok=True)
    for path in src.rglob('*'):
        if not path.is_file() or path.name in {'.gitkeep', '.DS_Store'}:
            continue
        if path.suffix.lower() in {'.pt', '.pth', '.ckpt', '.zip'}:
            continue
        if 'checkpoints' in path.parts or path.parts[:1] == ('temp',) or 'temp' in path.parts:
            continue
        target = dest / path.relative_to(src)
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(path, target)
        n += 1
    return n


n_results = copy_tree(repo / 'results', staging / 'results')
n_figures = copy_tree(repo / 'figures', staging / 'figures')

nb_candidates = [
    Path('/kaggle/working/__notebook__.ipynb'),
    Path('/kaggle/working/__notebook_source__.ipynb'),
    work / 'ams_skipgnn_kaggle_stage4.ipynb',
    repo / 'notebooks' / 'ams_skipgnn_kaggle_stage4.ipynb',
    work / 'ams_skipgnn_kaggle_runner.ipynb',
    repo / 'notebooks' / 'ams_skipgnn_kaggle_runner.ipynb',
]
nb_src = next((p for p in nb_candidates if p.is_file()), None)
if nb_src is not None:
    dest_nb = staging / 'notebooks' / 'ams_skipgnn_kaggle_stage4.ipynb'
    dest_nb.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(nb_src, dest_nb)

files = sorted(p.relative_to(staging).as_posix() for p in staging.rglob('*') if p.is_file())
manifest = {
    'created_utc': datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ'),
    'stage': os.environ.get('STAGE', '4'),
    'cwd': str(repo),
    'n_result_files': n_results,
    'n_figure_files': n_figures,
    'notebook_source': str(nb_src) if nb_src else None,
    'files': files,
    'import_map': {
        'results/': 'results/',
        'figures/': 'figures/',
        'notebooks/ams_skipgnn_kaggle_stage4.ipynb': 'notebooks/kaggle_stage4_ddi_ppi_gdi_executed.ipynb',
    },
}
try:
    import torch
    manifest['torch'] = torch.__version__
    manifest['cuda'] = bool(torch.cuda.is_available())
    if torch.cuda.is_available():
        manifest['gpu'] = torch.cuda.get_device_name(0)
except Exception:
    pass
(staging / 'MANIFEST.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
(staging / 'IMPORT.txt').write_text(
    'Drop this zip at the repo root and tell the assistant.\n'
    'It will merge results/, sync figures/, archive the executed notebook, and leave the repo delivery-ready.\n'
    'Command: python scripts/import_kaggle_bundle.py --src ams_skipgnn_kaggle_bundle.zip '
    '--archive-notebook notebooks/kaggle_stage4_ddi_ppi_gdi_executed.ipynb --make-figures\n',
    encoding='utf-8',
)

zip_path = work / 'ams_skipgnn_kaggle_bundle.zip'
if zip_path.exists():
    zip_path.unlink()
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for path in staging.rglob('*'):
        if path.is_file():
            zf.write(path, path.relative_to(staging).as_posix())
shutil.rmtree(staging, ignore_errors=True)

print('DOWNLOAD THIS FILE:', zip_path)
print('bytes', zip_path.stat().st_size)
print('files', len(files))
print('KAGGLE STAGE 4 COMPLETE — download only ams_skipgnn_kaggle_bundle.zip')
